In [ ]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [ ]:
# Import Packages
from Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from Input_Output_Rxn_Networks.CRNGenerator import CRNGenerator, CRNCompletor
from Input_Output_Rxn_Networks.ParameterSequenceGenerator import map_index_to_parameter
from Input_Output_Rxn_Networks.ReactionSequenceGenerator import map_indices_to_reactions
from copy import deepcopy
import gymnasium as gym
import torch
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import types
import time
import pathos.multiprocessing as mp
from itertools import product
from pathos.multiprocessing import ProcessingPool as Pool
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import multiprocess.context as ctx
ctx._force_start_method('fork')

In [ ]:
# Make sure to import comet_ml before torch or other libraries
import comet_ml
from pytorch_lightning.loggers import CometLogger

# Replace with your real API key
api_key = "o77J6VCMDamustkfJuMXZ2jdV"

# Initialize CometLogger with updated parameters
logger = CometLogger(
    api_key=api_key,
    project="your-project-name",        # Use `project` instead of `project_name`
    workspace="maurice-filo"     # Ensure this workspace exists on Comet
)

# Get the underlying Comet experiment object
experiment = logger.experiment

# Log some test metrics
for epoch in range(5):
    loss = 0.5 / (epoch + 1)
    accuracy = epoch * 0.1

    experiment.log_metric("loss", loss, step=epoch)
    experiment.log_metric("accuracy", accuracy, step=epoch)

# Ensure metrics get uploaded before finishing
experiment.end()

In [ ]:
# Select Device
device = torch.device("cuda" if 
torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Flags
save_flag = True                            # Save the agent checkpoint
load_flag = False                           # Load the agent checkpoint 
train_flag = True                           # Train the agent
save_filename = 'agent_checkpoint1.pth'
load_filename = 'agent_checkpoint1.pth'

In [ ]:
# Hyperparameters
N_CPUs = os.cpu_count()                 # Number of CPUs to use for parallel processing 
N_grid = 100                            # Number of grid points for the parameter range
N_samples = 32*N_CPUs                   # Number of samples to generate for Monte Carlo estimations
entropy_weight = 100                      # Initial Entropy weight for the loss function
entropy_update_coefficient = 0.9        # Coefficient for updating the entropy weight
entropy_schedule = 20                   # Number of epochs before updating the entropy weight
minimum_entropy_weight = 1           # Minimum entropy weight
num_epochs = 200                        # Number of epochs to train the model
learning_rate = 1e-5                    # Learning rate for the optimizer   
t_final = 150                           # Final time for the simulation

risk = 0.8
risk_update = 0.00
max_risk = 1.00
risk_schedule = 20

In [ ]:
# Construct an IOCRN modeling a Gene Expression process controlled by the AIF controller
species_labels = ['X_1', 'X_2', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2', 'u_3']
stoichiometry_reactants = np.array([[0, 1, 1, 0, 0, 0, 0], [0, 0, 0, 1, 0, 1, 0], [1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1]], dtype=np.int8)
stoichiometry_products = np.array([[1, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 1, 0], [1, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 1, 0]], dtype=np.int8)
k = 1; k_1 = 1; gamma_1 = 1; gamma_2 = 0.1; mu = 10; theta = 1; eta = 0.1
parameters = np.array([k, k_1, gamma_1, gamma_2, mu, theta, eta], dtype=np.float32)
input_influence_matrix = np.array([[0, 0, 0, 1, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0]], dtype=np.int8)
outputs = np.array([2], dtype=np.int8)
IOCRN_AIF = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
IOCRN_AIF.print_reactions()

# Transient Response for the AIF Controller
nums = [0.5, 1, 1.5]
u = np.array(list(product(nums, repeat=len(nums))), dtype=np.float32)
initial_condition = np.array([0, 0, 0, 0], dtype=np.float32)
time_horizon = np.linspace(0, t_final, 1000, dtype=np.float32)

tic = time.time()
outputs_AIF_transient = IOCRN_AIF.transient_response(u, initial_condition, time_horizon)
toc = time.time()
print('Simulation Time: %.2f' % (toc - tic))

plt.figure(figsize=(20, 6))
plt.subplot(1, 1, 1)
for i in range(len(outputs_AIF_transient)):
    plt.plot(time_horizon, outputs_AIF_transient[i])
plt.xlabel('Time')
plt.ylabel('Output')
plt.title('Transient Response')

print(np.linalg.norm(10 - outputs_AIF_transient[0]) / np.sqrt(len(outputs_AIF_transient[0])))

In [ ]:
# Construct the CRN Generator and Completor
num_species = 4
num_reactions = 2
num_inputs = 1

width = 256

# Define RSG_Attributes
class RSG_Attributes:
	LSTM_hidden_size = width
	FFNN_hidden_size = [width, width, width]
	FFNN_num_layers = [3, 3, 3]
	weight = [None, None, None]

# Define PSG_Attributes
class PSG_Attributes:
	LSTM_hidden_size = width
	FFNN_hidden_size = width
	FFNN_num_layers = 3
	weight = None

In [ ]:
# Design the loss function
def Performance_Metric(r, y):
    # return np.linalg.norm(r - y) / np.sqrt(len(y)) # + np.abs(y[-1] - r) 
    y = np.transpose(y, (1, 0, 2))
    weight = np.ones(y.shape[0])
    weight[(len(weight)//5)*4:] = weight[(len(weight)//5)*4:]*2
    weight[:(len(weight)//5)] = weight[:(len(weight)//5)]*0.25

    # reshape weight to match the shape of y
    weight = np.repeat(weight, y.shape[1]*y.shape[2]).reshape(y.shape[0], y.shape[1], y.shape[2])

    return (weight*np.abs(r - y)).mean()

def SS_Metric(r, y_ss):
    return np.abs(r - y_ss)

def unzip(params):
    out = []
    for i in range(len(params[0])):
        out.append([ p[i] for p in params ])
    return out

def _compute_single_loss(crn, inputs, initial_condition, time_horizon, r, threshold=1000):
    y, x = crn.transient_response(inputs, initial_condition, time_horizon, return_states=True)
    y = np.minimum(y, threshold)
    y = np.maximum(y,0)
    y[np.isnan(y)] = threshold
    y[np.isinf(y)] = threshold
    return Performance_Metric(r, y)

In [ ]:
# Create a batched Environment
class CRNEnv(gym.Env):
    """
    Custom Environment that follows gym interface
    This is the basic environment for CRNs.
    """
    def __init__(self, CRN_template, max_num_reactions):
        super(CRNEnv, self).__init__()
        self.CRN_template = CRN_template
        self.action_space = gym.spaces.Dict({
            'reactants space': gym.spaces.Discrete(self.CRN_template.get_complexes_range()),
            'products space': gym.spaces.Discrete(self.CRN_template.get_complexes_range()),
            'input influence space': gym.spaces.Discrete(self.CRN_template.num_inputs + 1),
            'rate constant space': gym.spaces.Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32)
        })
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(self.CRN_template.num_species,), dtype=np.float32)
        self.state = deepcopy(self.CRN_template)
        self.num_added_reactions = 0
        self.max_num_reactions = max_num_reactions

    def reset(self):
        self.state = deepcopy(self.CRN_template)
        self.num_added_reactions = 0
        return self.state

    def step(self, action):
        if self.state.num_unknown_parameters > 0:
            self.state.set_next_unknown_parameter(action)
        else:
            self.state.add_reaction(action)
            self.num_added_reactions += 1
            
        if self.num_added_reactions < self.max_num_reactions:
            done = False 
        else:
            done = True  
        
        info = {} 

        return self.state, done, info

    def render(self, mode='human'):
        if mode == 'human':
            self.state.print_reactions()
        else:
            pass        

# Helper functions (outside the class)
def reset_env(env):
    output = env.reset()
    return env, output

def step_env(env, action):
    output = env.step(action)
    return env, output

class VecEnv():
    def __init__(self, envs, N_CPUs=os.cpu_count()):
        self.envs = envs
        self.N_CPUs = N_CPUs
        self.pool = Pool(N_CPUs)

    def reset(self):
        return [env.reset() for env in self.envs]

    def step(self, actions):
        tic_step = time.time()
        output = [env.step(action) for env,action in zip(self.envs, actions)]
        toc_step = time.time()
        print('Step Time: %.2f' % (toc_step - tic_step))
        return output
    
    def get_reward(self, routine):
        tic_reward = time.time()
        rewards = self.pool.map(routine, [e.state for e in self.envs])
        toc_reward = time.time()
        print('Reward Time: %.2f' % (toc_reward - tic_reward))
        return rewards
    
    def gather(self):
        return [env.state for env in self.envs]

    def close(self):
        self.pool.close()
        self.pool.join()

class SerialVecEnv():
    def __init__(self, envs):
        self.envs = envs

    def reset(self):
        return [env.reset() for env in self.envs]

    def step(self, actions):
        tic_simulation = time.time()
        output = [env.step(action) for env,action in zip(self.envs, actions)]
        toc_simulation = time.time()
        print('Simulation Time: %.2f' % (toc_simulation - tic_simulation))
        return output

In [ ]:
# Construct Agent
class RecurrentAgent:
    def __init__(self, env, completor, learning_rate=1e-3):
        self.env = env
        self.completor = completor
        self.queue = []
        self.last_logP = None
        self.last_entropy = None
        self.optimizer = torch.optim.Adam(self.completor.parameters(), lr=learning_rate)

    def act(self):
        tic_forward = time.time()
        if len(self.queue) == 0 or len(self.queue[0]) == 0:
            batched_action_set, total_logP, total_entropy = self.completor()
            self.queue = batched_action_set
            self.last_logP = total_logP
            self.last_entropy = total_entropy

        action = [lst.pop(0) for lst in self.queue]
        toc_forward = time.time()
        print('Forward Time: %.2f' % (toc_forward - tic_forward))
        return action
    
    def update(self, rewards):
        tic_backward = time.time()
        self.optimizer.zero_grad()
        loss_for_each_sample = torch.tensor(rewards, requires_grad=False).to(self.completor.device)
        loss_for_each_sample_comp = loss_for_each_sample # if you need to do some postprocessing of it
        entropy_mean = torch.mean(self.last_entropy)
        total_entropy = torch.mean(self.last_entropy, dim=0)
        print('Last Entropy', self.last_entropy.shape)

        loss_for_gradient =  (loss_for_each_sample_comp.detach() * self.last_logP) - entropy_weight * entropy_mean - (entropy_weight * total_entropy.detach() * self.last_logP)
        top_k = torch.topk(loss_for_each_sample, int(N_samples * (1.-risk)), largest=False).indices # - entropy_weight*total_entropy

        best = loss_for_each_sample[top_k[0]]
        worst = loss_for_each_sample[top_k[-1]]
        avg = loss_for_each_sample[top_k].mean()

        loss_for_gradient = torch.mean(loss_for_gradient[top_k]).backward()

        #clip gradients
        # torch.nn.utils.clip_grad_norm_(MyCRN_Generator.parameters(), 0.01)
        self.optimizer.step()
        toc_backward = time.time()
        print('Backward Time: %.2f' % (toc_backward - tic_backward))
        print('best', best.item(), 'worst', worst.item(), 'avg', avg.item())

In [ ]:
# Construct the reward function
def compute_reward(state):
    nums = [0.5, 1, 1.5]
    u = np.array(list(product(nums, repeat=len(nums))), dtype=np.float32)
    initial_condition = np.array([0, 0, 0, 0], dtype=np.float32)
    time_horizon = np.linspace(0, t_final, 1000, dtype=np.float32)
    r = u[:,2] * state.parameters[4]
    return _compute_single_loss(state, u, initial_condition, time_horizon, r, threshold=1000)

# Construct the parameter grid
param_1 = np.linspace(0.01, 20, N_grid, dtype=np.float32)
param_2 = np.linspace(0.01, 20, N_grid, dtype=np.float32)
param_3 = np.linspace(0.01, 20, N_grid, dtype=np.float32)
parameter_grid = np.stack([param_1, param_2, param_3], axis=0)
MyCRN_Generator = CRNGenerator(num_species, num_reactions, 1, num_inputs, parameter_grid, N_samples, RSG_Attributes, PSG_Attributes, device=device).to(device)
MyCRN_Completor = CRNCompletor(num_species, num_reactions, 1, num_inputs, parameter_grid, N_samples, RSG_Attributes, PSG_Attributes, device=device).to(device)
IOCRN_AIF.set_parameters(5, np.nan)
vec_env = VecEnv([CRNEnv(IOCRN_AIF, num_reactions) for _ in range(N_samples)], N_CPUs=N_CPUs)
# vec_env = SerialVecEnv([CRNEnv(IOCRN_AIF, num_reactions) for _ in range(N_samples)])
recurrent_agent = RecurrentAgent(vec_env, MyCRN_Completor)

In [ ]:
# Load the agent model
if load_flag:
    recurrent_agent = torch.load(load_filename, weights_only=False)

# Train the agent model
if train_flag:
    for epoch in tqdm(range(num_epochs)):
        obs = vec_env.reset()
        rewards = []
        for i in range(3):
            # Take an action
            action = recurrent_agent.act()
            
            # Step the environment
            output = vec_env.step(action)

        # Get rewards
        rewards = vec_env.get_reward(compute_reward)

        # Update the agent
        recurrent_agent.update(rewards)

# Render the environment
vec_env.envs[0].render()

In [ ]:
# Test the model
obs = vec_env.reset()
rewards = []
for i in range(3):
    # Take an action
    action = recurrent_agent.act()
    
    # Step the environment
    output = vec_env.step(action)

# Get rewards
rewards = vec_env.get_reward(compute_reward)

# Gather the CRNs
crns = vec_env.gather()

# Sort the CRNs by rewards
sorted_crns_rewards = sorted(zip(crns, rewards), key=lambda x: x[1])

# Plot the transient responses
outputs = []
for i in tqdm(range(20)):
    outputs_transient = sorted_crns_rewards[i][0].transient_response(u, initial_condition, time_horizon)
    outputs.append(outputs_transient)

# Print the best CRN
print('CRN 1:')
sorted_crns_rewards[0][0].print_reactions()
print('CRN 2:')
sorted_crns_rewards[1][0].print_reactions()
print('CRN 3:')
sorted_crns_rewards[2][0].print_reactions()

In [ ]:
colors = ['r', 'g', 'b']
for j in range(10):
    for i in range(len(u)):
        plt.plot(time_horizon, outputs[j][i], alpha=0.1)

In [ ]:
# Save the model
if save_flag:
    torch.save(recurrent_agent, save_filename)